# Introduction to Neuroinformatics
## Exercise session 1: What is neuroinformatics?

Welcome to the first exercise session of the Introduction to Neuroinformatics course. Neuroinformatics is the study of the brain through a computational lens, and this course applies that lens at several very different levels: single membranes, whole populations of neurons, and the electronic hardware built to imitate them.

This notebook is a tour of those three levels, one per question:

1. **How does a single neuron compute?** We build a neuron out of ion channels and watch a spike come out of it.
2. **How do we read a computation out of the activity of many neurons?** We look at a population that solves a decision task, first one neuron at a time and then all at once.
3. **How do we build with it?** We look at what the brain's way of computing costs in energy, and why anyone would want to copy it in silicon.

You are not asked to write code today. Every cell below is meant to be run as it is, so that the rest of the course has something concrete to attach to. Two written assignments are the only work.

**It is fine if much of this does not make sense yet.** None of it is assumed knowledge, and none of it is meant to be mastered today. The terms in **bold** are the ones a lecture will define properly later in the semester, each in its own session, so treat this notebook as a map rather than as a lesson: run the cells, get a feel for what the objects look like, and come back to it as the course goes on. Section 4 says which lecture picks up which thread. The same figures should read very differently the second time.

One warning up front: this is **not** a machine learning course. We train no networks and optimize no losses here. The question throughout is how brains compute, and what that buys us.

This notebook is designed to be run on Google Colab. If you wish to save your progress, click on the "Copy to Drive" button.

---

# Table of contents

* [Packages](#packages)
* [1: Modelling the computational unit](#1)
  * [1.1: The membrane as an electrical circuit](#1.1)
  * [1.2: The action potential](#1.2)
  * [1.3: The F-I curve](#1.3)
* [2: Reading computation out of a population](#2)
  * [2.1: The task](#2.1)
  * [2.2: What single units look like](#2.2)
  * [2.3: The population state space](#2.3)
* [3: Building with it](#3)
  * [3.1: The world as spike trains](#3.1)
  * [3.2: How much of the network is doing anything](#3.2)
  * [3.3: The energy argument](#3.3)
* [4: Where this course goes](#4)
* [5: Further Readings](#further)

---

# Packages <a name="packages"></a>

The first cell below checks whether we are running on Google Colab. Everything that is specific to Colab — installing packages, creating directories, downloading the helper modules and the data — is guarded behind that check, so the notebook does not fail on its first cell if you open it somewhere else. Colab is the platform this course supports and tests.

**The download is large.** The population activity used in Section 2 is about **97 MB**, nearly all of it a single file of single-unit responses, so the setup cell takes tens of seconds rather than the second or two you may be used to. Start it now and read on while it runs.

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print(f"IN_COLAB: {IN_COLAB}")

In [ ]:
if IN_COLAB:
    !pip install -U ipywidgets

    !mkdir -p utils_ex1
    !mkdir -p utils_ex7
    !mkdir -p utils_ex7/figures
    !mkdir -p utils_ex7/data_ctx
    !mkdir -p utils_ex13

    # Section 1: the Hodgkin-Huxley neuron.
    !wget -P utils_ex1/ https://github.com/ManteLab/Iton_notebooks_public/raw/refs/heads/main/utils_ex1/hh.py

    # Section 2: the context-dependent task and one of notebook 7's four models.
    !wget -P utils_ex7/ https://github.com/ManteLab/Iton_notebooks_public/raw/refs/heads/main/utils_ex7/ctx_dependent_utils.py
    !wget -P utils_ex7/figures/ https://github.com/ManteLab/Iton_notebooks_public/raw/refs/heads/main/utils_ex7/figures/task.png
    !wget -P utils_ex7/data_ctx/ https://github.com/ManteLab/Iton_notebooks_public/raw/refs/heads/main/utils_ex7/data_ctx/task_conditions.mat
    !wget -P utils_ex7/data_ctx/ https://github.com/ManteLab/Iton_notebooks_public/raw/refs/heads/main/utils_ex7/data_ctx/betResp_choice_inputmot_inputcol_modela.mat
    !wget -P utils_ex7/data_ctx/ https://github.com/ManteLab/Iton_notebooks_public/raw/refs/heads/main/utils_ex7/data_ctx/betResp_inputmot_choice_inputcol_modela.mat
    !wget -P utils_ex7/data_ctx/ https://github.com/ManteLab/Iton_notebooks_public/raw/refs/heads/main/utils_ex7/data_ctx/betResp_inputmot_inputcol_choice_modela.mat
    !wget -P utils_ex7/data_ctx/ https://github.com/ManteLab/Iton_notebooks_public/raw/refs/heads/main/utils_ex7/data_ctx/Regmod_units_modela.mat
    !wget -P utils_ex7/data_ctx/ https://github.com/ManteLab/Iton_notebooks_public/raw/refs/heads/main/utils_ex7/data_ctx/Rtot_units_modela.mat
    !wget -P utils_ex7/data_ctx/ https://github.com/ManteLab/Iton_notebooks_public/raw/refs/heads/main/utils_ex7/data_ctx/bmat1_modela.mat
    !wget -P utils_ex7/data_ctx/ https://github.com/ManteLab/Iton_notebooks_public/raw/refs/heads/main/utils_ex7/data_ctx/bmat2_modela.mat
    !wget -P utils_ex7/data_ctx/ https://github.com/ManteLab/Iton_notebooks_public/raw/refs/heads/main/utils_ex7/data_ctx/betas_norm_modela.mat
    !wget -P utils_ex7/data_ctx/ https://github.com/ManteLab/Iton_notebooks_public/raw/refs/heads/main/utils_ex7/data_ctx/pcaResp_modela.mat
    !wget -P utils_ex7/data_ctx/ https://github.com/ManteLab/Iton_notebooks_public/raw/refs/heads/main/utils_ex7/data_ctx/per_var_modela.mat

    # Section 3: the spiking version of notebook 13's dataset.
    !wget -P utils_ex13/ https://github.com/ManteLab/Iton_notebooks_public/raw/refs/heads/main/utils_ex13/ann_data.py
    !wget -P utils_ex13/ https://github.com/ManteLab/Iton_notebooks_public/raw/refs/heads/main/utils_ex13/snn_data.py

In [ ]:
from ipywidgets import Dropdown, interact

from utils_ex1.hh import iplot_action_potential, iplot_fi_curve, plot_spike_sparsity
from utils_ex7.ctx_dependent_utils import (
    load_context_dependent_models,
    plot_projections_2d,
    plot_psths,
    show_task,
)
from utils_ex13.snn_data import YinYangPoissonDataset, plot_sample_slider

---

# 1. Modelling the computational unit <a name="1"></a>

A neuron is usually first drawn as a **threshold gate**: sum the weighted inputs, compare the sum to a threshold, emit a one or a zero. That picture is old — McCulloch and Pitts published it in 1943 — and it is the picture digital computing inherited, by way of von Neumann. It is also not what a neuron is.

A real neuron is a **dynamical system**. It is a patch of membrane with a few kinds of voltage-dependent ion channel in it, and its state evolves continuously in time. There is no line in it that compares a sum to a threshold.

So this section does the modelling the other way round. We write the patch of membrane down as an electrical circuit, integrate it, and watch the things we would otherwise have *assumed* — an all-or-nothing spike, a threshold, an input-output curve — come out of the biophysics on their own. Lectures 2 to 4 and notebooks 2 to 4 do this properly; here we only look at the result.

---

## 1.1. The membrane as an electrical circuit <a name="1.1"></a>

The membrane holds charge apart, so it is a **capacitor**. The channels in it pass current, so they are **conductances**, each in series with a battery whose voltage is set by the concentration gradient of its ion. Hodgkin and Huxley measured those conductances in the squid giant axon in 1952 and found that two of them, sodium and potassium, depend on the membrane voltage itself. That feedback is the entire mechanism.

Conservation of current across the patch gives one equation:

$$
C_m \frac{dV}{dt} = I_\text{ext} - \bar{g}_\text{Na}\, m^3 h \,(V - E_\text{Na}) - \bar{g}_\text{K}\, n^4 \,(V - E_\text{K}) - g_\text{L} \,(V - E_\text{L})
$$

The three terms subtracted from the injected current are the sodium, potassium and leak currents. Each is a conductance times a **driving force**, the distance of the voltage from that ion's **reversal potential**. The factors $m$, $h$ and $n$ are the **gating variables**: numbers between 0 and 1 giving the fraction of gates of each kind that are open. They are what makes the conductances voltage-dependent, and each follows its own first-order equation,

$$
\frac{dx}{dt} = \alpha_x(V)\,(1 - x) - \beta_x(V)\,x, \qquad x \in \{m, h, n\}
$$

where the opening and closing rates $\alpha_x$ and $\beta_x$ are fixed functions of the voltage, fitted to measurements.

That is the whole **Hodgkin-Huxley model**: four coupled ordinary differential equations. Voltages are in millivolts, time in milliseconds, and injected current is per unit membrane area, in uA/cm^2 — the units on the axes below. Everything that follows is these four equations integrated forward in time.

---

## 1.2. The action potential <a name="1.2"></a>

Run the cell below and move the slider. It injects a constant current into the model from the previous section and plots the **membrane potential** in black, with the three gating variables on the right-hand axis. Each redraw re-integrates the equations and takes a fraction of a second.

Things worth noticing:

1. **Below about 7 uA/cm^2, nothing happens.** The voltage settles at a new resting level and stays there.
2. **Above that, the neuron spikes, and the spikes are all the same size.** Doubling the current does not give you a bigger **action potential**; it gives you more of them. This is where the phrase **all-or-nothing** comes from.
3. **Watch the order of the gates.** $m$ (red, sodium activation) rises almost vertically at the start of each spike; $h$ (blue, sodium inactivation) falls a little later and shuts the sodium current off, which ends the spike; $n$ (green, potassium activation) rises slowly and drags the voltage back down past rest.
4. **There is no threshold parameter in the model.** Nowhere did we write down a **firing threshold** — and yet the neuron plainly has one. It is a property of the dynamics, not a number in the code.

In [ ]:
iplot_action_potential()

---

## 1.3. The F-I curve <a name="1.3"></a>

If the spikes are all identical, then the only thing a neuron can vary is **how often** it fires. The **F-I curve** — firing frequency against injected current — is therefore the compact summary of a neuron's input-output behaviour, and you will meet it again in almost every later session.

The cell below builds one. Because each point on the curve means simulating the neuron all over again, it does not redraw as you drag: set the two sliders, then **click "Run sweep"**, and give it a couple of seconds.

Things worth noticing:

1. **Below the rheobase the rate is exactly zero.** The **rheobase** is the smallest steady current that makes the neuron fire at all, and the dashed line marks it.
2. **At the rheobase the rate jumps.** It does not rise smoothly from zero — the neuron's first sustained firing is already around 50 to 60 Hz. A Hodgkin-Huxley neuron cannot fire arbitrarily slowly, which is a genuine and slightly surprising property of the real squid axon.
3. **Above that the curve rises and then flattens.** The flattening is the **refractory period**: after a spike, sodium inactivation ($h$) and potassium activation ($n$) need time to recover, and no amount of extra current can make the neuron fire before they have.

In [ ]:
iplot_fi_curve()

---

# 2. Reading computation out of a population <a name="2"></a>

Section 1 had one neuron and we knew its mechanism exactly, because we wrote it. Real experiments are the opposite: hundreds of neurons, recorded while an animal does something, and no access to the mechanism at all. The second question of this course is what to do with that. **Given the activity of many neurons, how do you find the computation in it?**

The example here is the one Lectures 7 to 9 keep returning to, from Mante et al. (2013). A monkey watches a patch of dots that are both **moving** in some direction and **coloured** some mixture of red and green. On each trial a cue tells it which of the two to attend to, and it reports either the direction or the dominant colour. Both features are present on every trial. Only the **context** says which one matters, and the same physical stimulus can require opposite answers in the two contexts.

What we plot below are simulated populations of 500 units that solve that task. Notebook 7 gives you four of them — they produce the same behaviour by visibly different means — and asks you to work out which is which. Here we look at one of them, dataset **a**, and do not ask how it works.

---

## 2.1. The task <a name="2.1"></a>

The figure below lays out the task and the behaviour: the stimulus, the two contexts, and the **psychometric curves** showing that the animal's choices follow the relevant feature and ignore the irrelevant one.

In [ ]:
show_task()

---

## 2.2. What single units look like <a name="2.2"></a>

First, the obvious thing to do with 500 units: look at them one at a time. The cell below picks six at random and plots their condition-averaged responses — one **PSTH**, a peri-stimulus time histogram, per condition — sorted four ways: by choice, by motion strength in the motion context, by colour strength in the colour context, and by context. Re-run it to draw six different units.

This is the point of the section, so look at the plots before reading on. A few units are clean and interpretable. Most are not: they mix choice, motion, colour and context together in proportions that differ from unit to unit, and no single one of them is "the choice neuron" or "the context neuron". The computation is really there — the population solves the task — but it is not legible one unit at a time.

In [ ]:
# About 100 MB is read from disk here, so this cell takes a few seconds.
model_a = load_context_dependent_models("model_a")

In [ ]:
plot_psths(model_a)

---

## 2.3. The population state space <a name="2.3"></a>

Now the same data all at once. Treat the population's response at one moment as a **point in a 500-dimensional space**, one axis per unit — the population's **state space** — and a trial becomes a **trajectory** through it. That space is far too big to look at, so we project it down to two dimensions and plot the trajectories there. The dropdowns below choose how:

- **PCA** (`pca` in the dropdown) picks the directions of greatest variance, without being told anything about the task. Each panel plots one **principal component** against another, one trajectory per condition.
- **TDR** (`tdr`), targeted dimensionality reduction, instead uses directions found by regressing each unit's response against the task variables, so that the axes *mean* something: choice, motion input, colour input. Lecture 7 and notebook 7 derive it.

The **Context** dropdown selects which context's trials are shown. It applies to the PCA view; the TDR view already shows both contexts, motion in the top row and colour in the bottom.

What to look for: in the TDR projection there is a **choice axis** along which the trajectories separate by the eventual answer, and the relevant input pushes the population along it while the irrelevant one does not. That separation is nowhere to be found in Section 2.2. It exists in the population, not in the units.

In [ ]:
# The order in which the TDR axes are orthogonalized; notebook 7 explains why
# this is the right one.
order_orthogonalization = ["choice", "input_motion", "input_color"]


@interact(
    method=Dropdown(options=["pca", "tdr"], value="pca", description="Projection:"),
    context=Dropdown(options=["motion", "color"], value="motion", description="Context:"),
)
def show_state_space(method, context):
    """Project the population response onto two of its axes."""
    plot_projections_2d(model_a, method, order_orthogonalization, context)

---

> **Assignment 1**
>
> This task can also be solved by an ordinary computer program, and it is solved by a brain. Compare the two implementations.
>
> 1. Describe in words how you would program a conventional computer to solve the context-dependent task. What does the context variable do in your program?
> 2. In the state space you just plotted, context does *not* gate the inputs: both of them enter the population, and the dynamics select the relevant one along the choice axis. Describe how that differs from your answer to question 1.
> 3. What does the population implementation buy you — think about noisy inputs, and about what happens when you lose a few units — and what does it cost you? (Hint: in your program from question 1 there is a variable holding the context, and you can print it. Here there is no such variable.)

**Solution:**

---

---

# 3. Building with it <a name="3"></a>

The third question is the one the course ends on. **If the brain computes like that, can we build hardware that computes like that too?** This is **neuromorphic engineering**, the tradition this institute works in, and the argument for it is not that brains are accurate — they are slow, noisy and variable — but that they are unreasonably cheap to run.

To see where the cheapness comes from we need the brain's signalling in its own currency: not numbers, but **spikes** — discrete events, sparse in time. So we first look at what a piece of the world looks like when it has been turned into spike trains, then count how much of the network is doing anything at any given moment, and only then talk about watts.

---

## 3.1. The world as spike trains <a name="3.1"></a>

The dataset below is the one notebook 13 uses. Each sample is a point $(x, y)$ in the unit square, and its four features — $x$, $y$, $1-x$ and $1-y$ — are encoded as **Poisson spike trains** over a window of 200 timesteps: a large feature value means a high **firing rate**, a small one means a low rate. That is one of the standard ways of getting ordinary data into a spiking system, and it is roughly what a sensory neuron does with a stimulus.

Move the sliders to choose a point. The left panel shows the four spike trains it becomes, drawn as a **raster plot**: one row per input unit, one tick per spike. The right panel shows where the point sits in the dataset. Each move regenerates the spike trains and redraws both panels, so give it about a second, and use "Resample" to draw a fresh set of spikes for the same point — the encoding is random, so the same input never gives exactly the same spikes twice.

In [ ]:
# 500 samples is plenty to see the structure, and takes well under a second.
spike_dataset = YinYangPoissonDataset(size=500, seed=42)
plot_sample_slider(spike_dataset)

---

## 3.2. How much of the network is doing anything <a name="3.2"></a>

Now count the spikes instead of looking at them. The left panel below is one sample's raster again; the right panel is the fraction of units active at each timestep, averaged over 200 samples.

The number is around **5%**. Out of 800 unit-timesteps in a sample, some forty carry an event and the rest carry nothing at all.

That is the whole architectural argument in one figure. A clocked digital system evaluates every unit at every timestep whether or not anything happened, because that is what a clock does. An **event-driven** substrate — biological or neuromorphic — does work only where and when there is an event, so a 5% activity rate means roughly 5% of the work. Nothing about the data changed; only the assumption that computation must be scheduled by a clock.

In [ ]:
plot_spike_sparsity(spike_dataset, n_samples=200)

---

## 3.3. The energy argument <a name="3.3"></a>

Put numbers on it. A human brain has on the order of 86 billion neurons and more than a hundred trillion synapses, and it runs on about **20 W** — less than the lamp on your desk. The comparison that closes Lecture 1 sets that against a GPT-4-scale system, with a broadly comparable count of units and connections, drawing **30 to 70 MW**: a factor of roughly a million, and enough that the practical constraint is a power plant. Further down the scale, a bee gets by on a brain of 1 mg and about 960,000 neurons, at something like $10^{-15}$ J per spike, and still navigates, forages and learns.

Where does the difference come from? Not mainly from arithmetic. In a conventional accelerator, fetching an operand from DRAM costs on the order of a thousand times more energy than the multiply-accumulate it feeds, because the memory and the processor are separate places and the data has to be moved between them — the **von Neumann bottleneck**. A brain has no such separation: the **synapse** both stores the weight and performs the operation, in the same physical spot. That gives the two design principles the last lectures build on —

1. **Exploit physical space.** Use one physical circuit per neuron and per synapse instead of time-multiplexing one fast processor across all of them, and co-locate memory with computation.
2. **Let time represent itself.** Match the circuits' **time constants** to the timescales of the signals, and let the computation run in real time rather than in simulated steps.

— and a spiking substrate is what makes both affordable, because, as Section 3.2 showed, almost nothing is happening at any given moment.

None of this makes a brain a better computer than a computer. It makes it a spectacularly efficient one, and the claim this course opens with is that **the most efficient machine that can learn, anywhere we know of, is the biological brain** — so it is worth understanding how the first one is built before trying to build the second.

---

# 4. Where this course goes <a name="4"></a>

Each stop on the tour is a thread that runs through several lectures and their notebooks.

| This notebook | Lectures | Notebooks |
|---|---|---|
| **Section 1** — the neuron as a dynamical system | L2 Neurons, resting potentials, spikes; L3 Passive membrane and action potentials; L4 The synapse | 2, 3, 4 |
| **Section 2** — computation as population geometry | L7 Computations in neural populations; L8 and L9 Computations through dynamics | 7, 8, 9 |
| **Section 3** — spikes, energy and hardware | L12 Learning in spiking neural networks; L13 Neural circuits and electronic circuits | 12, 13 |

The lectures the tour skips are not detours. Sessions 5 and 6 stay with the single neuron and ask what it computes; sessions 10 and 11 are about how connections change with experience, in biology and in artificial networks. Three stops is a tour, not a syllabus.

Use the table to come back here. When a lecture picks up one of the threads, re-run the section that matches it — the figures will not have changed, but what you can see in them will have.

Notebook 2 starts on the first of these next week, from ions and a bare RC circuit.

---

> **Assignment 2**
>
> Think of a problem from your own field — whatever you were working on before you walked into this course.
>
> 1. How would you solve it algorithmically today?
> 2. Would understanding how a brain solves an analogous problem plausibly give you a better algorithm? Say why or why not. "No" is a perfectly good answer if you can defend it.
> 3. Name one thing you would expect a brain to be *worse* at than a conventional computer.
>
> There is no model answer to this one, here or in the solutions notebook. Bring what you wrote to the exercise session and compare it with your neighbour's.

---

---

# 5. Further Readings <a name="further"></a>

There is no assigned reading for this session — the lecture covers the material you need. The references below are for going further, grouped by the questions this notebook raises.

**Section 1 — the neuron as a dynamical system.**

- Hodgkin & Huxley (1952), "A quantitative description of membrane current and its application to conduction and excitation in nerve," *J. Physiol.* — the Hodgkin-Huxley model, which Section 1 simulates.
- Dayan, P. & Abbott, L. F. (2001), *Theoretical Neuroscience*, Chapters 5 and 6 — the standard treatment of the membrane, the single-compartment neuron and the models built on it.

**Section 2 — reading computation out of a population.**

- Mante, Sussillo, Shenoy & Newsome (2013), "Context-dependent computation by recurrent dynamics in prefrontal cortex," *Nature* — the study whose task and models Section 2 plots: selection and integration of a contextually relevant input, and the approximate line attractor that supports it.
- Shenoy, Sahani & Churchland (2013), *Annual Review of Neuroscience* — survey of neural population dynamics and motor preparation, for the wider case that the population is the right level of description.

**Section 3 — spikes, energy and hardware.**

- Indiveri & Liu (2015), "Memory and information processing in neuromorphic systems," *Proceedings of the IEEE* — the review closest to this course's own perspective: mixed-signal analog neuromorphic circuits, in which the physics of the transistor is used to emulate neural dynamics directly rather than to simulate them digitally.
- Roy, Jaiswal & Panda (2019), "Towards spike-based machine intelligence with neuromorphic computing," *Nature* — the standard review of the case for spike-based computation, covering algorithms and hardware together.

**Getting set up.**

- [The Python Tutorial](https://docs.python.org/3/tutorial/) and the [NumPy absolute beginner's guide](https://numpy.org/doc/stable/user/absolute_beginners.html) — if you have not written Python before, these two are enough to follow every notebook in this course.
- [Overview of Colab features](https://colab.research.google.com/notebooks/basic_features_overview.ipynb) — how the notebook interface itself works: running cells, saving your own copy to Drive, and restarting the runtime when something goes wrong.

---

# End of this exercise session